# 🧠 Module 14 Lab — Agent Memory Systems & RAG Architecture

**Mục tiêu:** Xây dựng production-grade RAG system cho LoanBot v3.1

Trong lab này, bạn sẽ implement:
1. `LoanBotChunker` — Semantic chunking cho regulations và loan decisions
2. `InMemoryVectorStore` — Mock vector store với cosine similarity search
3. `RAGRetriever` — Multi-stage: embed → search → rerank → top-K
4. `ConversationMemory` — Progressive summarization với token budget
5. `SemanticCache` — Cosine-similarity based response cache
6. `LoanBotRAGPipeline` — End-to-end RAG pipeline với quality gate

## Section 1: Setup và Mock Data

In [ ]:
import hashlib
import math
import random
import statistics
import time
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

print('✅ Imports OK')

# ─── Mock Regulation Documents ───
REGULATIONS = [
    {'id': 'TT39-D15', 'circular': 'TT39/2016/TT-NHNN', 'article': 'Điều 15',
     'title': 'Tỷ lệ thế chấp',
     'text': 'Điều 15. Tỷ lệ thế chấp tối thiểu. Đối với khoản vay có tài sản thế chấp là bất động sản, giá trị tài sản thế chấp phải đạt tối thiểu 120% giá trị khoản vay theo giá thị trường tại thời điểm thẩm định. Ngân hàng phải thuê đơn vị thẩm định độc lập cho các khoản vay từ 500 triệu VNĐ trở lên.',
     'applies_to': 'real_estate_collateral', 'effective_date': '2016-12-01'},

    {'id': 'TT39-D20', 'circular': 'TT39/2016/TT-NHNN', 'article': 'Điều 20',
     'title': 'Điều kiện thu nhập người vay',
     'text': 'Điều 20. Điều kiện về thu nhập. Người vay phải chứng minh thu nhập ổn định từ việc làm hoặc kinh doanh. Tỷ lệ nợ trên thu nhập (DTI) không vượt quá 50% thu nhập hàng tháng. Thời gian làm việc tại đơn vị hiện tại tối thiểu 12 tháng đối với nhân viên, và 24 tháng đối với chủ doanh nghiệp.',
     'applies_to': 'income_verification', 'effective_date': '2016-12-01'},

    {'id': 'NHNN-CB01', 'circular': 'CV5627/NHNN-TTGSNH', 'article': 'Khoản 3',
     'title': 'Ngưỡng điểm tín dụng',
     'text': 'Khoản 3. Phân loại khách hàng theo điểm tín dụng. Nhóm A (điểm 720–850): khách hàng tốt, ưu tiên phê duyệt. Nhóm B (điểm 620–719): khách hàng trung bình, cần xem xét thêm. Nhóm C (điểm 560–619): khách hàng có rủi ro, cần HITL review. Nhóm D (dưới 560): từ chối hoặc yêu cầu bảo lãnh.',
     'applies_to': 'credit_scoring', 'effective_date': '2023-06-01'},

    {'id': 'NHNN-AML', 'circular': 'TT20/2019/TT-NHNN', 'article': 'Điều 8',
     'title': 'Phòng chống rửa tiền',
     'text': 'Điều 8. Nghi ngờ giao dịch bất thường. Ngân hàng phải báo cáo giao dịch khi xác suất gian lận vượt 30% theo hệ thống scoring nội bộ. Khoản vay từ 300 triệu VNĐ trở lên phải kiểm tra danh sách PEP (Politically Exposed Persons) và danh sách đen quốc tế.',
     'applies_to': 'fraud_aml', 'effective_date': '2019-08-01'},

    {'id': 'NHNN-HITL', 'circular': 'TT41/2021/TT-NHNN', 'article': 'Điều 12',
     'title': 'Phê duyệt thủ công cho khoản vay lớn',
     'text': 'Điều 12. Phê duyệt thủ công bắt buộc. Khoản vay có giá trị từ 500 triệu VNĐ trở lên phải có sự phê duyệt của ít nhất một cán bộ tín dụng có thẩm quyền. Hệ thống AI/tự động hóa không được ra quyết định cuối cùng đối với các khoản vay thuộc nhóm này.',
     'applies_to': 'hitl_mandatory', 'effective_date': '2021-09-01'},

    {'id': 'INTERNAL-P1', 'circular': 'FINTECHCORP-POL-001', 'article': 'Section 3',
     'title': 'Chính sách tín dụng nội bộ',
     'text': 'Section 3. Tỷ lệ phê duyệt mục tiêu. FinTech Corp duy trì tỷ lệ phê duyệt 60-65% để đảm bảo cân bằng giữa tăng trưởng và rủi ro. Portfolio risk score trung bình không vượt quá 0.40. Nhóm khách hàng Nhóm D (điểm < 560) tự động từ chối trừ khi có bảo lãnh từ khách hàng Nhóm A.',
     'applies_to': 'internal_policy', 'effective_date': '2024-01-01'},
]

# ─── Mock Historical Loan Decisions ───
LOAN_HISTORY = [
    {'loan_id': 'FC-2023-0891', 'credit_score': 718, 'amount_vnd': 200_000_000,
     'income_vnd': 22_000_000, 'decision': 'APPROVE', 'risk_score': 0.28,
     'reason': 'Good credit profile, stable income, DTI=0.38', 'outcome': 'performing',
     'date': '2023-06-15'},
    {'loan_id': 'FC-2024-001', 'credit_score': 720, 'amount_vnd': 200_000_000,
     'income_vnd': 25_000_000, 'decision': 'APPROVE', 'risk_score': 0.22,
     'reason': 'Excellent credit, low fraud risk (0.05), DTI=0.27', 'outcome': 'performing',
     'date': '2024-01-20'},
    {'loan_id': 'FC-2023-1240', 'credit_score': 582, 'amount_vnd': 180_000_000,
     'income_vnd': 14_000_000, 'decision': 'REJECT', 'risk_score': 0.61,
     'reason': 'Borderline score, income insufficient for DTI threshold', 'outcome': 'N/A',
     'date': '2023-10-01'},
    {'loan_id': 'FC-2024-004-R1', 'credit_score': 640, 'amount_vnd': 400_000_000,
     'income_vnd': 28_000_000, 'decision': 'REJECT', 'risk_score': 0.52,
     'reason': 'Regulatory flag, borderline score 640, high amount requires additional verification', 'outcome': 'N/A',
     'date': '2024-01-10'},
]

print(f'✅ Loaded {len(REGULATIONS)} regulations, {len(LOAN_HISTORY)} historical decisions')

## Section 2: Chunker và Mock Vector Store

In [ ]:
@dataclass
class Chunk:
    chunk_id:    str
    content:     str
    metadata:    Dict
    parent_id:   Optional[str]
    token_count: int
    embedding:   Optional[List[float]] = None  # populated after embedding

class LoanBotChunker:
    """Semantic chunker cho regulations và loan decisions."""

    def chunk_regulation(self, reg: Dict) -> Chunk:
        content = f"{reg['article']}. {reg['title']}\n{reg['text']}"
        chunk_id = hashlib.md5(content.encode()).hexdigest()[:12]
        return Chunk(
            chunk_id=chunk_id, content=content,
            metadata={
                'type': 'regulation', 'circular': reg['circular'],
                'article': reg['article'], 'applies_to': reg['applies_to'],
                'effective_date': reg['effective_date'],
            },
            parent_id=reg['circular'],
            token_count=len(content.split()) * 4 // 3,
        )

    def chunk_loan_decision(self, decision: Dict) -> Chunk:
        content = (
            f"Loan Decision Record\n"
            f"ID: {decision['loan_id']} | Date: {decision['date']}\n"
            f"Customer profile: credit_score={decision['credit_score']} "
            f"income={decision['income_vnd']:,} VNĐ\n"
            f"Loan amount: {decision['amount_vnd']:,} VNĐ\n"
            f"Decision: {decision['decision']} | Risk score: {decision['risk_score']}\n"
            f"Reason: {decision['reason']}\n"
            f"Outcome: {decision.get('outcome', 'N/A')}"
        )
        chunk_id = hashlib.md5(content.encode()).hexdigest()[:12]
        return Chunk(
            chunk_id=chunk_id, content=content,
            metadata={'type': 'loan_history', **decision},
            parent_id=None,
            token_count=len(content.split()) * 4 // 3,
        )

# ─── Mock Embedder ───
class MockEmbedder:
    """Deterministic mock embedder: same text → same vector (for reproducibility)."""

    DIM = 64  # reduced dim for lab speed

    def embed(self, text: str) -> List[float]:
        """Hash-based deterministic embedding in DIM dimensions."""
        seed = int(hashlib.md5(text.encode()).hexdigest(), 16) % (2**31)
        rng = random.Random(seed)
        vec = [rng.gauss(0, 1) for _ in range(self.DIM)]
        # normalize
        norm = math.sqrt(sum(x**2 for x in vec))
        return [x/norm for x in vec]

    def cosine_sim(self, a: List[float], b: List[float]) -> float:
        dot = sum(x*y for x,y in zip(a,b))
        na  = math.sqrt(sum(x**2 for x in a))
        nb  = math.sqrt(sum(x**2 for x in b))
        return dot / (na * nb + 1e-10)

# ─── In-Memory Vector Store ───
class InMemoryVectorStore:
    """Mock Qdrant: store chunks with embeddings, support similarity search + metadata filter."""

    def __init__(self):
        self._chunks: Dict[str, Chunk] = {}
        self.embedder = MockEmbedder()

    def upsert(self, chunk: Chunk):
        chunk.embedding = self.embedder.embed(chunk.content)
        self._chunks[chunk.chunk_id] = chunk

    def search(self, query: str, top_k: int = 20,
               filter_metadata: Optional[Dict] = None) -> List[Tuple[Chunk, float]]:
        query_vec = self.embedder.embed(query)
        scored = []
        for chunk in self._chunks.values():
            # Apply metadata filter
            if filter_metadata:
                match = all(chunk.metadata.get(k) == v for k,v in filter_metadata.items())
                if not match: continue
            sim = self.embedder.cosine_sim(query_vec, chunk.embedding)
            scored.append((chunk, sim))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def count(self) -> int:
        return len(self._chunks)

# ── Index all documents ──
chunker = LoanBotChunker()
store   = InMemoryVectorStore()

for reg in REGULATIONS:
    store.upsert(chunker.chunk_regulation(reg))

for decision in LOAN_HISTORY:
    store.upsert(chunker.chunk_loan_decision(decision))

print(f'✅ Vector store indexed: {store.count()} chunks')
print(f'   {len(REGULATIONS)} regulation chunks + {len(LOAN_HISTORY)} loan history chunks')

## Section 3: RAG Retriever với Reranking

In [ ]:
class MockReranker:
    """Mock cross-encoder reranker: scores query-chunk pairs."""

    def __init__(self):
        self.embedder = MockEmbedder()

    def rerank(self, query: str, candidates: List[Tuple[Chunk, float]],
               top_n: int = 5) -> List[Tuple[Chunk, float]]:
        """Mock: use longer bi-encoder with slightly different hash for re-scoring."""
        query_vec = self.embedder.embed(query + ' [RERANK]')  # different representation
        rescored = []
        for chunk, orig_score in candidates:
            # Combine original score with reranker score (weighted)
            rerank_score = self.embedder.cosine_sim(query_vec, chunk.embedding)
            combined = 0.3 * orig_score + 0.7 * rerank_score
            rescored.append((chunk, combined))
        rescored.sort(key=lambda x: x[1], reverse=True)
        return rescored[:top_n]

class RAGRetriever:
    """Multi-stage retriever: search top-20 → rerank → top-5."""

    def __init__(self, store: InMemoryVectorStore, top_k_retrieve: int = 20,
                 top_k_final: int = 5):
        self.store     = store
        self.reranker  = MockReranker()
        self.top_k_retrieve = top_k_retrieve
        self.top_k_final    = top_k_final

    def retrieve(self, query: str,
                 filter_metadata: Optional[Dict] = None) -> List[Tuple[Chunk, float]]:
        t0 = time.monotonic()

        # Stage 1: Initial retrieval
        candidates = self.store.search(query, top_k=self.top_k_retrieve, filter_metadata=filter_metadata)

        # Stage 2: Rerank
        final = self.reranker.rerank(query, candidates, top_n=self.top_k_final)

        latency_ms = int((time.monotonic()-t0)*1000)
        return final, latency_ms

# ── Demo retrieval for each customer ──
retriever = RAGRetriever(store)
print('=== RAG Retrieval Demo ===')

test_queries = [
    ('FC-2024-001 — score 720, amount 200M', 'loan approval criteria credit score 720 collateral requirements standard'),
    ('FC-2024-002 — score 580 borderline',   'borderline credit score 580 income verification DTI requirements'),
    ('FC-2024-003 — fraud probability 0.72', 'fraud detection AML blacklist high risk loan rejection'),
    ('FC-2024-004 — regulatory flag',        'mandatory human review HITL large loan regulatory compliance'),
]

for loan_id, query in test_queries:
    results, latency = retriever.retrieve(query)
    print(f"\n{loan_id}")
    print(f"  Query: '{query[:50]}...'")
    print(f"  Retrieved {len(results)} chunks in {latency}ms:")
    for i, (chunk, score) in enumerate(results[:3]):  # show top-3
        ctype = chunk.metadata.get('type', 'unknown')
        clabel = chunk.metadata.get('article', chunk.metadata.get('loan_id', '?'))
        print(f"  [{i+1}] score={score:.3f} type={ctype} ref={clabel}")

## Section 4: Conversation Memory với Progressive Summarization

In [ ]:
@dataclass
class TokenBudget:
    total_limit:      int = 200_000
    system_reserved:  int = 8_000
    rag_budget:       int = 12_000
    history_budget:   int = 20_000
    loan_data_budget: int = 4_000
    output_reserved:  int = 4_096

    @property
    def available_buffer(self) -> int:
        used = (self.system_reserved + self.rag_budget +
                self.history_budget + self.loan_data_budget + self.output_reserved)
        return self.total_limit - used  # safety buffer: ~151,904 tokens

class ConversationMemory:
    """Rolling window memory with auto-summarization (mock — no real API call)."""

    def __init__(self, budget: TokenBudget, session_id: str):
        self.budget     = budget
        self.session_id = session_id
        self.messages:  List[Dict] = []
        self.summary:   str = ''
        self._token_est = 0
        self.compress_count = 0

    def _estimate_tokens(self, text: str) -> int:
        return len(text) // 4

    def add_message(self, role: str, content: str):
        self.messages.append({'role': role, 'content': content})
        self._token_est += self._estimate_tokens(content)
        if self._token_est > self.budget.history_budget * 0.8:
            self._compress()

    def _compress(self):
        if len(self.messages) <= 5:
            return
        to_summarize = self.messages[:-5]
        self.compress_count += 1

        # Mock summarization (real: call claude-haiku)
        key_points = []
        for m in to_summarize:
            words = m['content'].split()
            key_points.append(' '.join(words[:10]))
        self.summary = f"[Compressed summary #{self.compress_count}]: " + '; '.join(key_points[:3]) + '...'

        self.messages  = self.messages[-5:]
        self._token_est = (self._estimate_tokens(self.summary) +
                           sum(self._estimate_tokens(m['content']) for m in self.messages))

    def build_context(self) -> List[Dict]:
        messages = []
        if self.summary:
            messages.append({'role': 'user',      'content': f'[Session summary]: {self.summary}'})
            messages.append({'role': 'assistant', 'content': 'Noted the previous conversation context.'})
        messages.extend(self.messages)
        return messages

    def stats(self) -> Dict:
        return {
            'message_count': len(self.messages),
            'token_estimate': self._token_est,
            'budget_used_pct': round(self._token_est / self.budget.history_budget * 100, 1),
            'has_summary': bool(self.summary),
            'compress_count': self.compress_count,
        }

# ── Demo: Simulate long loan review session ──
budget = TokenBudget()
memory = ConversationMemory(budget, 'SESSION-FC-2024-004')

conversation_turns = [
    ('user',      'Tôi muốn xem xét lại hồ sơ FC-2024-004 của Phạm Thị D.'),
    ('assistant', 'FC-2024-004: Phạm Thị D, score 645, khoản vay 450M VNĐ. Hồ sơ này đã bị từ chối lần trước do regulatory flag và score borderline. RiskAgent đề nghị HITL Level 3.'),
    ('user',      'Score 645 thuộc nhóm C theo quy định NHNN. Chị ấy có thể bổ sung tài sản thế chấp không?'),
    ('assistant', 'Theo Điều 15 TT39, tỷ lệ thế chấp tối thiểu 120% = 540M VNĐ. Chị D cần bổ sung bất động sản trị giá ≥ 540M. Hiện chị D có nhà tại Q.7 trị giá ~600M VNĐ — đủ điều kiện.'),
    ('user',      'Regulatory flag từ đâu? Chị có trong danh sách đen không?'),
    ('assistant', 'Regulatory flag vì chị D có giao dịch với đối tác bị kiểm soát trong danh sách OFAC. Tuy nhiên giao dịch đó đã được giải trình đầy đủ vào tháng 3/2024. Không có trong danh sách đen PEP hay NHNN.'),
    ('user',      'Income của chị có đủ không? DTI ratio?'),
    ('assistant', 'Thu nhập 30M VNĐ/tháng, khoản vay 450M/60 tháng = 7.5M/tháng. DTI = 7.5/30 = 25% — dưới ngưỡng 50% theo Điều 20. Thu nhập đủ điều kiện.'),
    ('user',      'Quyết định của tôi: phê duyệt có điều kiện — cần nộp thêm giấy tờ giải trình OFAC trong 5 ngày.'),
    ('assistant', 'Ghi nhận quyết định: CONDITIONAL_APPROVE. Điều kiện: nộp văn bản giải trình OFAC trong 5 ngày làm việc. Nếu không nộp → tự động REJECT. Tôi sẽ tạo SLA reminder.'),
    ('user',      'Bây giờ xem hồ sơ FC-2024-002 của Trần Thị B.'),
    ('assistant', 'FC-2024-002: Trần Thị B, score 580, khoản vay 150M VNĐ, income 12M VNĐ. Score 580 = Nhóm C. Fraud prob 0.18 = MEDIUM. DTI = 12.5%. RiskAgent: REVIEW.'),
]

print('=== Conversation Memory Demo ===')
for role, content in conversation_turns:
    memory.add_message(role, content)
    s = memory.stats()
    compress_flag = ' ← COMPRESSED' if s['compress_count'] > 0 and s['message_count'] <= 5 else ''
    print(f"  [{role:10s}] tokens={s['token_estimate']:5d} ({s['budget_used_pct']:5.1f}% budget) "
          f"msgs={s['message_count']}{compress_flag}")

print(f"\n📊 Final stats: {memory.stats()}")
print(f"\n📝 Context built: {len(memory.build_context())} messages")
if memory.summary:
    print(f"💾 Summary: {memory.summary[:120]}...")

## Section 5: Semantic Cache + Quality Gate

In [ ]:
@dataclass
class CachedEntry:
    query:       str
    embedding:   List[float]
    response:    str
    hits:        int = 0
    created_at:  float = field(default_factory=time.time)
    ttl_seconds: int = 86_400  # 24 hours

    @property
    def is_expired(self) -> bool:
        return time.time() - self.created_at > self.ttl_seconds

class SemanticCache:
    """Cache using cosine similarity to find near-identical queries."""

    def __init__(self, sim_threshold: float = 0.95):
        self.threshold = sim_threshold
        self.embedder  = MockEmbedder()
        self._cache:   List[CachedEntry] = []
        self.hits = 0; self.misses = 0

    def lookup(self, query: str) -> Optional[str]:
        query_vec = self.embedder.embed(query)
        for entry in self._cache:
            if entry.is_expired:
                continue
            sim = self.embedder.cosine_sim(query_vec, entry.embedding)
            if sim >= self.threshold:
                entry.hits += 1
                self.hits += 1
                return entry.response
        self.misses += 1
        return None

    def store(self, query: str, response: str):
        embedding = self.embedder.embed(query)
        self._cache.append(CachedEntry(query=query, embedding=embedding, response=response))

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0

@dataclass
class RAGQualityGate:
    faithfulness_min: float = 0.85
    relevancy_min:    float = 0.80
    precision_min:    float = 0.75

    def evaluate(self, query: str, answer: str, contexts: List[str]) -> Dict:
        # Mock RAGAS scores — derived from query/answer/context overlap
        query_words   = set(query.lower().split())
        answer_words  = set(answer.lower().split())
        context_words = set(' '.join(contexts).lower().split())

        faithfulness      = min(1.0, len(answer_words & context_words) / (len(answer_words) + 1) * 2.5)
        answer_relevancy  = min(1.0, len(answer_words & query_words) / (len(query_words) + 1) * 3.0)
        context_precision = min(1.0, len(context_words & query_words) / (len(context_words) + 1) * 10.0)

        passed = (
            faithfulness >= self.faithfulness_min and
            answer_relevancy >= self.relevancy_min and
            context_precision >= self.precision_min
        )
        return {
            'passed':     passed,
            'action':     'serve' if passed else 'hitl_fallback',
            'scores': {
                'faithfulness':      round(faithfulness, 3),
                'answer_relevancy':  round(answer_relevancy, 3),
                'context_precision': round(context_precision, 3),
            }
        }

# ── Demo ──
cache = SemanticCache(sim_threshold=0.95)
gate  = RAGQualityGate()

# Simulate queries — some repeated
queries = [
    ('loan approval criteria score 720 collateral', 'FC-2024-001: Score 720 (Nhóm A). Theo Điều 15 TT39, collateral 120% required. Amount 200M → collateral 240M needed. DTI 0.27 < 50%. Decision: APPROVE.'),
    ('fraud detection AML blacklist high risk',     'FC-2024-003: Fraud probability 0.72 > 0.30 threshold. Blacklist match=True. Theo TT20/2019 Điều 8, phải báo cáo. Decision: REJECT.'),
    ('loan approval criteria score 725 collateral', None),  # Should HIT cache (similar to query 1)
    ('HITL mandatory large loan regulatory',        'FC-2024-004: 450M > 500M threshold. Theo TT41/2021 Điều 12, bắt buộc HITL. Regulatory flag present. Level 3 Credit Committee.'),
    ('fraud detection high probability AML review', None),  # Should HIT cache (similar to query 2)
]

print('=== Semantic Cache Demo ===')
for query, mock_response in queries:
    cached = cache.lookup(query)
    if cached:
        print(f"  HIT  [{query[:45]:45s}] → served from cache")
    else:
        # Generate response (mock)
        response = mock_response or f'Generated response for: {query[:30]}'
        # Quality gate
        contexts = [r.text for r in REGULATIONS[:3]]
        qg = gate.evaluate(query, response, contexts)
        if qg['passed']:
            cache.store(query, response)
            print(f"  MISS [{query[:45]:45s}] → generated + cached (faith={qg['scores']['faithfulness']:.2f})")
        else:
            print(f"  MISS [{query[:45]:45s}] → HITL (faith={qg['scores']['faithfulness']:.2f} < {gate.faithfulness_min})")

print(f"\n📊 Cache stats: hits={cache.hits} misses={cache.misses} hit_rate={cache.hit_rate:.1%}")
print(f"   (Target: ≥40% hit rate for production LoanBot)")

## Section 6: LoanBot RAG Pipeline — End-to-End

In [ ]:
CUSTOMERS = {
    'FC-2024-001': {'name': 'Nguyễn Văn A', 'credit_score': 720, 'loan_amount_vnd': 200_000_000,
                    'monthly_income_vnd': 25_000_000, 'fraud_probability': 0.05,
                    'regulatory_flag': False, 'collateral_type': 'real_estate'},
    'FC-2024-002': {'name': 'Trần Thị B',   'credit_score': 580, 'loan_amount_vnd': 150_000_000,
                    'monthly_income_vnd': 12_000_000, 'fraud_probability': 0.18,
                    'regulatory_flag': False, 'collateral_type': 'vehicle'},
    'FC-2024-003': {'name': 'Lê Văn C',     'credit_score': 400, 'loan_amount_vnd': 80_000_000,
                    'monthly_income_vnd': 8_000_000,  'fraud_probability': 0.72,
                    'regulatory_flag': False, 'collateral_type': 'none'},
    'FC-2024-004': {'name': 'Phạm Thị D',   'credit_score': 645, 'loan_amount_vnd': 450_000_000,
                    'monthly_income_vnd': 30_000_000, 'fraud_probability': 0.28,
                    'regulatory_flag': True,  'collateral_type': 'real_estate'},
}

class LoanBotRAGPipeline:
    """Full RAG pipeline: retrieve → augment → generate (mock) → quality gate."""

    def __init__(self):
        self.retriever = RAGRetriever(store)
        self.cache     = SemanticCache()
        self.gate      = RAGQualityGate()
        self.calls     = 0
        self.cache_hits = 0

    def _build_query(self, loan_id: str, customer: Dict) -> str:
        cs = customer['credit_score']
        group = 'nhóm A tốt' if cs >= 720 else 'nhóm C borderline' if cs >= 560 else 'nhóm D xấu'
        parts = [f"thẩm định khoản vay {customer['loan_amount_vnd']//1_000_000}M VNĐ",
                 f"điểm tín dụng {cs} {group}",
                 f"thu nhập {customer['monthly_income_vnd']//1_000_000}M VNĐ",
                 f"tài sản thế chấp {customer['collateral_type']}"]
        if customer['fraud_probability'] > 0.30:
            parts.append('nghi ngờ gian lận AML cần kiểm tra')
        if customer['regulatory_flag']:
            parts.append('cần phê duyệt thủ công bắt buộc')
        return ' '.join(parts)

    def _mock_generate(self, loan_id: str, customer: Dict,
                       retrieved_chunks: List[Tuple[Chunk, float]]) -> str:
        """Mock RiskAgent response referencing retrieved regulations."""
        cs = customer['credit_score']
        fp = customer['fraud_probability']
        dti = customer['loan_amount_vnd'] / (customer['monthly_income_vnd'] * 60)

        refs = [f"{c.metadata.get('article', c.metadata.get('loan_id', '?'))}" for c,_ in retrieved_chunks[:3]]

        if fp > 0.70:
            decision = 'REJECT'; reason = f'Xác suất gian lận {fp:.0%} > 30% ngưỡng theo TT20/2019 Điều 8. Blacklist match.'
        elif cs < 560:
            decision = 'REJECT'; reason = f'Điểm tín dụng {cs} < 560 (Nhóm D). Theo CV5627 Khoản 3: từ chối.'
        elif customer['regulatory_flag']:
            decision = 'HITL'; reason = f'Regulatory flag. Theo TT41/2021 Điều 12: khoản vay cần phê duyệt thủ công.'
        elif cs >= 700 and fp < 0.15 and dti < 0.4:
            decision = 'APPROVE'; reason = f'Điểm {cs} (Nhóm A), DTI={dti:.2f} < 50%, fraud {fp:.0%} = LOW.'
        else:
            decision = 'REVIEW'; reason = f'Cần xem xét thêm: score {cs}, fraud {fp:.0%}.'

        return (f"LoanBot RAG Assessment — {loan_id}\n"
                f"Quyết định: {decision}\nLý do: {reason}\n"
                f"Tham chiếu quy định: {', '.join(refs)}\n"
                f"DTI: {dti:.2f} | Fraud tier: {'HIGH' if fp>0.5 else 'MEDIUM' if fp>0.2 else 'LOW'}")

    def process(self, loan_id: str, customer: Dict) -> Dict:
        self.calls += 1
        query = self._build_query(loan_id, customer)

        # Check cache
        cached = self.cache.lookup(query)
        if cached:
            self.cache_hits += 1
            return {'loan_id': loan_id, 'source': 'cache', 'response': cached}

        # Retrieve
        results, latency_ms = self.retriever.retrieve(query)
        contexts = [c.content for c,_ in results]

        # Generate (mock)
        response = self._mock_generate(loan_id, customer, results)

        # Quality gate
        qg = self.gate.evaluate(query, response, contexts)
        if qg['passed']:
            self.cache.store(query, response)
        else:
            response = f'[HITL REQUIRED] Faithfulness={qg["scores"]["faithfulness"]:.2f} below threshold. Human review needed.'

        return {
            'loan_id': loan_id, 'source': 'generated',
            'response': response, 'quality': qg,
            'retrieved_chunks': len(results), 'retrieval_ms': latency_ms,
        }

# ── Run end-to-end ──
pipeline = LoanBotRAGPipeline()
print('=== LoanBot RAG Pipeline — End-to-End Demo ===')

for loan_id, customer in CUSTOMERS.items():
    result = pipeline.process(loan_id, customer)
    print(f"\n{'='*60}")
    print(f"{result['loan_id']} ({customer['name']}) — source: {result['source']}")
    print(result['response'])
    if 'quality' in result:
        print(f"\n  Quality: {result['quality']['scores']} → {result['quality']['action']}")
        print(f"  Retrieved: {result['retrieved_chunks']} chunks in {result['retrieval_ms']}ms")

print(f"\n📊 Pipeline Summary:")
print(f"  Total calls:  {pipeline.calls}")
print(f"  Cache hits:   {pipeline.cache_hits} ({pipeline.cache_hits/pipeline.calls:.0%})")

## Section 7: Thực hành — Metadata Filtering

**Bài tập:** Implement filtered retrieval: khi customer có `regulatory_flag=True`, chỉ retrieve regulation chunks có `applies_to='hitl_mandatory'` hoặc `applies_to='fraud_aml'`. Đồng thời, khi `credit_score < 560`, chỉ retrieve `applies_to='credit_scoring'`.

**Gợi ý:** Modify `LoanBotRAGPipeline.process()` để pass `filter_metadata` vào `retriever.retrieve()`.

In [ ]:
# ── Không gian thực hành ──
# TODO: Implement smart metadata filtering

def get_filter_for_customer(customer: Dict) -> Optional[Dict]:
    """Return metadata filter based on customer profile."""
    # TODO: implement
    # if customer['regulatory_flag']: return {'applies_to': 'hitl_mandatory'}
    # if customer['fraud_probability'] > 0.30: return {'applies_to': 'fraud_aml'}
    # if customer['credit_score'] < 560: return {'applies_to': 'credit_scoring'}
    return None  # no filter (baseline)

# Test: FC-2024-004 with regulatory_flag=True should only retrieve hitl/aml regulations
filter_test = get_filter_for_customer(CUSTOMERS['FC-2024-004'])
results_filtered, _ = retriever.retrieve(
    'mandatory HITL large loan regulatory compliance',
    filter_metadata=filter_test
)
print(f'Filtered retrieval: {len(results_filtered)} chunks')
for chunk, score in results_filtered[:3]:
    print(f"  [{score:.3f}] {chunk.metadata.get('applies_to', '?')} — {chunk.metadata.get('article', chunk.metadata.get('loan_id', '?'))}")
print('Implement get_filter_for_customer() to see targeted retrieval!')